# BioPetals Colab Quickstart

This notebook installs BioPetals from your fork and runs OpenBioLLM inference over the Petals network.

In [ ]:
# 1. Install base dependencies from the official package to avoid building C++ packages from source
%pip install -q petals

# 2. Clone your modded BioPetals repository
!rm -rf BioPetals
!git clone https://github.com/Pranesh950/BioPetals.git

# 3. Inform Python to use your modded code instead of the official library
import sys
sys.path.insert(0, '/content/BioPetals/src')

In [ ]:
import warnings
from transformers import AutoConfig

warnings.filterwarnings("ignore", message=".*HF_TOKEN.*", category=UserWarning)

MODEL_NAME = "aaditya/Llama3-OpenBioLLM-8B"
cfg = AutoConfig.from_pretrained(MODEL_NAME)
print("model_type:", cfg.model_type)
assert cfg.model_type == "llama", "This checkpoint is not llama-compatible for Petals."

In [ ]:
import torch
from petals.client import load_biology_model

PROMPT = "Explain CRISPR-Cas9 and how NHEJ can create insertion or deletion mutations."

try:
    tokenizer, model = load_biology_model(
        MODEL_NAME,
        max_retries=1,
        connect_timeout=5,
        request_timeout=120,
        show_route="inference",
    )

    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT}],
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = PROMPT

    inputs = tokenizer(prompt_text, return_tensors="pt")["input_ids"]

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=160,
            do_sample=True,
            temperature=0.2,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(outputs[0, inputs.shape[-1]:], skip_special_tokens=True).strip()
    print("=== Model output ===")
    print(answer)

except Exception as e:
    print("Could not run this biology model on the public swarm right now.")
    print("Reason:", repr(e))
    print("Retry later or run your own private swarm and host this model.")